# Microsoft Purview Unified Catalog - Data Quality Advanced Analysis

This notebook demonstrates advanced usage of the `pvw uc quality` commands with:
- Data visualization using matplotlib and seaborn
- Quality score trending and comparison
- Domain-level quality analytics
- Asset quality distribution analysis

## Prerequisites
- Purview CLI installed (`pip install pvw-cli`)
- Azure authentication configured
- Access to Purview Unified Catalog with Data Quality enabled
- Python packages: pandas, matplotlib, seaborn

In [ ]:
# Install required packages
!pip install pvw-cli pandas matplotlib seaborn -q

In [ ]:
import subprocess
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from typing import Dict, List, Any

# Configure visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Imports successful")

In [ ]:
# Configuration
ACCOUNT_ID = "your-purview-account"  # Replace with your Purview account ID

def run_quality_cmd(args: List[str]) -> Dict[str, Any]:
    """Execute pvw uc quality command and return parsed JSON output."""
    cmd = ["pvw", "uc", "quality"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error: {result.stderr}")
        return {}
    
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        print(f"Non-JSON output: {result.stdout}")
        return {}

print("✓ Helper functions loaded")

## Alternative: Using Python Client API Directly

In addition to CLI commands via subprocess, you can use the purviewcli Python client directly for programmatic access.

In [ ]:
# Import purviewcli Python client
from purviewcli.client import DataQuality
import os

# Initialize the client (uses Azure authentication automatically)
quality_client = DataQuality()

# Example 1: List business domains using Python API
print("Example 1: List Business Domains\\n")
domains_response = quality_client.list_business_domains({"--account": ACCOUNT_ID})
print(f"Status Code: {quality_client.status_code}")

if quality_client.status_code == 200:
    domains_api = domains_response.get("value", [])
    print(f"Found {len(domains_api)} domains via Python API\\n")
    for domain in domains_api[:3]:  # Show first 3
        print(f"  - {domain.get('name', 'N/A')} (ID: {domain.get('id', 'N/A')[:8]}...)")
else:
    print(f"Error: {domains_response}")

# Example 2: Get domain report using Python API
if domains_api:
    test_domain_id = domains_api[0].get("id")
    print(f"\\n\\nExample 2: Get Domain Report for {domains_api[0].get('name', 'Unknown')}\\n")
    
    report_response = quality_client.get_domain_report({
        "--account": ACCOUNT_ID,
        "--domain-id": test_domain_id
    })
    
    if quality_client.status_code == 200:
        print(f"Overall Quality Score: {report_response.get('overallQualityScore', 'N/A')}/100")
        print(f"Total Assets: {report_response.get('totalAssets', 0)}")
        print(f"Assessed Assets: {report_response.get('assessedAssets', 0)}")
        print(f"Freshness Score: {report_response.get('freshnessScore', 'N/A')}/100")
        print(f"Completeness Score: {report_response.get('completenessScore', 'N/A')}/100")
        print(f"Accuracy Score: {report_response.get('accuracyScore', 'N/A')}/100")
    else:
        print(f"Error: {report_response}")

    # Example 3: List domain data sources using Python API
    print(f"\\n\\nExample 3: List Data Sources for Domain\\n")
    sources_response = quality_client.list_domain_data_sources({
        "--account": ACCOUNT_ID,
        "--domain-id": test_domain_id
    })
    
    if quality_client.status_code == 200:
        sources = sources_response.get("value", [])
        print(f"Found {len(sources)} data sources")
        for source in sources[:5]:  # Show first 5
            print(f"  - {source.get('name', 'N/A')} (Type: {source.get('type', 'N/A')})")
    else:
        print(f"Error: {sources_response}")

    # Example 4: List domain assets using Python API with pagination
    print(f"\\n\\nExample 4: List Domain Assets with Pagination\\n")
    assets_response = quality_client.list_domain_assets({
        "--account": ACCOUNT_ID,
        "--domain-id": test_domain_id,
        "--top": "10",  # Limit to 10 results
        "--skip": "0"   # Skip first 0 (for pagination)
    })
    
    if quality_client.status_code == 200:
        assets = assets_response.get("value", [])
        print(f"Retrieved {len(assets)} assets (limited to 10)")
        for asset in assets[:5]:  # Show first 5
            print(f"  - {asset.get('name', 'N/A')} | Score: {asset.get('qualityScore', 'N/A')}")
    else:
        print(f"Error: {assets_response}")

print("\\n✓ Python API examples completed")

## 1. Domain Discovery and Overview (Using CLI Commands)

In [ ]:
# List all business domains
domains_data = run_quality_cmd(["domains", "--account", ACCOUNT_ID])
domains = domains_data.get("value", [])

# Create domains DataFrame
if domains:
    df_domains = pd.DataFrame([{
        "Name": d.get("name", "N/A"),
        "ID": d.get("id", "N/A"),
        "Description": d.get("description", "N/A")[:50] + "..." if d.get("description") else "N/A"
    } for d in domains])
    
    print(f"Found {len(domains)} business domains:\n")
    display(df_domains)
else:
    print("No domains found")

## 2. Quality Score Analysis by Domain

In [ ]:
# Collect quality scores for each domain
quality_scores = []

for domain in domains[:5]:  # Limit to first 5 domains for performance
    domain_id = domain.get("id")
    domain_name = domain.get("name", "Unknown")
    
    # Get domain report
    report = run_quality_cmd(["domain-report", "--account", ACCOUNT_ID, "--domainId", domain_id])
    
    if report:
        quality_scores.append({
            "Domain": domain_name,
            "ID": domain_id,
            "Total Assets": report.get("totalAssets", 0),
            "Assessed Assets": report.get("assessedAssets", 0),
            "Quality Score": report.get("overallQualityScore", 0),
            "Freshness Score": report.get("freshnessScore", 0),
            "Completeness Score": report.get("completenessScore", 0),
            "Accuracy Score": report.get("accuracyScore", 0)
        })

df_scores = pd.DataFrame(quality_scores)

if not df_scores.empty:
    print("Quality Scores by Domain:\n")
    display(df_scores)
else:
    print("No quality scores available")

In [ ]:
# Visualize quality scores comparison
if not df_scores.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Overall quality score comparison
    ax1 = axes[0]
    bars = ax1.barh(df_scores['Domain'], df_scores['Quality Score'], color='steelblue')
    ax1.set_xlabel('Quality Score', fontweight='bold')
    ax1.set_title('Overall Quality Score by Domain', fontsize=14, fontweight='bold')
    ax1.set_xlim(0, 100)
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax1.text(width + 2, bar.get_y() + bar.get_height()/2, 
                f'{width:.1f}', ha='left', va='center', fontweight='bold')
    
    # Score dimensions comparison
    ax2 = axes[1]
    score_dims = df_scores[['Domain', 'Freshness Score', 'Completeness Score', 'Accuracy Score']]
    score_dims_melted = score_dims.melt(id_vars='Domain', var_name='Dimension', value_name='Score')
    
    sns.barplot(data=score_dims_melted, x='Domain', y='Score', hue='Dimension', ax=ax2)
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_xlabel('Domain', fontweight='bold')
    ax2.set_title('Quality Dimensions by Domain', fontsize=14, fontweight='bold')
    ax2.legend(title='Dimension', loc='lower right')
    ax2.set_ylim(0, 100)
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

## 3. Asset Quality Distribution

In [ ]:
# Analyze asset distribution for a specific domain
if domains:
    target_domain = domains[0]
    domain_id = target_domain.get("id")
    domain_name = target_domain.get("name", "Unknown")
    
    print(f"\nAnalyzing assets for domain: {domain_name}\n")
    
    # Get domain assets
    assets_data = run_quality_cmd(["assets", "--account", ACCOUNT_ID, "--domainId", domain_id, "--limit", "100"])
    assets = assets_data.get("value", [])
    
    if assets:
        # Create assets DataFrame
        df_assets = pd.DataFrame([{
            "Asset Name": a.get("name", "N/A"),
            "Type": a.get("assetType", "N/A"),
            "Quality Score": a.get("qualityScore", 0),
            "Data Source": a.get("dataSourceName", "N/A"),
            "Last Assessed": a.get("lastAssessedTime", "N/A")
        } for a in assets])
        
        print(f"Retrieved {len(assets)} assets\n")
        display(df_assets.head(10))
        
        # Quality score statistics
        print("\nQuality Score Statistics:")
        print(df_assets['Quality Score'].describe())
    else:
        print("No assets found for this domain")
        df_assets = pd.DataFrame()

In [ ]:
# Visualize asset quality distribution
if not df_assets.empty and 'Quality Score' in df_assets.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Histogram of quality scores
    ax1 = axes[0]
    ax1.hist(df_assets['Quality Score'], bins=20, color='skyblue', edgecolor='black')
    ax1.set_xlabel('Quality Score', fontweight='bold')
    ax1.set_ylabel('Frequency', fontweight='bold')
    ax1.set_title(f'Quality Score Distribution\n{domain_name}', fontsize=12, fontweight='bold')
    ax1.axvline(df_assets['Quality Score'].mean(), color='red', linestyle='--', label=f'Mean: {df_assets["Quality Score"].mean():.1f}')
    ax1.legend()
    
    # Box plot by asset type
    ax2 = axes[1]
    if len(df_assets['Type'].unique()) > 1:
        df_assets.boxplot(column='Quality Score', by='Type', ax=ax2)
        ax2.set_xlabel('Asset Type', fontweight='bold')
        ax2.set_ylabel('Quality Score', fontweight='bold')
        ax2.set_title(f'Quality Score by Asset Type\n{domain_name}', fontsize=12, fontweight='bold')
        plt.sca(ax2)
        plt.xticks(rotation=45, ha='right')
    else:
        ax2.text(0.5, 0.5, 'Insufficient asset types for comparison', 
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'Quality Score by Asset Type\n{domain_name}', fontsize=12, fontweight='bold')
    
    # Top/Bottom performers
    ax3 = axes[2]
    top_5 = df_assets.nlargest(5, 'Quality Score')
    bottom_5 = df_assets.nsmallest(5, 'Quality Score')
    combined = pd.concat([top_5, bottom_5])
    
    colors = ['green' if score >= df_assets['Quality Score'].median() else 'red' 
             for score in combined['Quality Score']]
    
    y_pos = range(len(combined))
    ax3.barh(y_pos, combined['Quality Score'], color=colors, alpha=0.7)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels([name[:30] + '...' if len(name) > 30 else name 
                         for name in combined['Asset Name']], fontsize=8)
    ax3.set_xlabel('Quality Score', fontweight='bold')
    ax3.set_title(f'Top & Bottom Performers\n{domain_name}', fontsize=12, fontweight='bold')
    ax3.set_xlim(0, 100)
    
    plt.tight_layout()
    plt.show()

## 4. Data Source Quality Analysis

In [ ]:
# Analyze quality by data source
if not df_assets.empty and 'Data Source' in df_assets.columns:
    # Group by data source
    source_quality = df_assets.groupby('Data Source').agg({
        'Quality Score': ['mean', 'min', 'max', 'count']
    }).round(2)
    
    source_quality.columns = ['Avg Score', 'Min Score', 'Max Score', 'Asset Count']
    source_quality = source_quality.sort_values('Avg Score', ascending=False)
    
    print(f"Quality Analysis by Data Source:\n")
    display(source_quality)
    
    # Visualize
    if len(source_quality) > 1:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = range(len(source_quality))
        width = 0.35
        
        ax.bar([i - width/2 for i in x], source_quality['Avg Score'], width, 
              label='Average Score', color='steelblue', alpha=0.8)
        ax.bar([i + width/2 for i in x], source_quality['Asset Count'] * 5, width,  # Scale for visibility
              label='Asset Count (×5)', color='coral', alpha=0.8)
        
        ax.set_xlabel('Data Source', fontweight='bold')
        ax.set_ylabel('Score / Count', fontweight='bold')
        ax.set_title(f'Data Source Quality Overview - {domain_name}', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(source_quality.index, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 5. Quality Alerts Analysis

In [ ]:
# Get quality alerts for the domain
if domains:
    alerts_data = run_quality_cmd(["alerts", "--account", ACCOUNT_ID, "--domainId", domain_id])
    alerts = alerts_data.get("value", [])
    
    if alerts:
        df_alerts = pd.DataFrame([{
            "Alert Name": a.get("name", "N/A"),
            "Severity": a.get("severity", "N/A"),
            "Status": a.get("status", "N/A"),
            "Asset": a.get("assetName", "N/A"),
            "Created": a.get("createdTime", "N/A")
        } for a in alerts])
        
        print(f"\nQuality Alerts for {domain_name}:\n")
        display(df_alerts)
        
        # Alert summary
        print("\nAlert Summary:")
        print(f"Total Alerts: {len(alerts)}")
        if 'Severity' in df_alerts.columns:
            print("\nBy Severity:")
            print(df_alerts['Severity'].value_counts())
        if 'Status' in df_alerts.columns:
            print("\nBy Status:")
            print(df_alerts['Status'].value_counts())
        
        # Visualize alerts
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        if 'Severity' in df_alerts.columns and len(df_alerts['Severity'].unique()) > 0:
            severity_counts = df_alerts['Severity'].value_counts()
            axes[0].pie(severity_counts.values, labels=severity_counts.index, autopct='%1.1f%%',
                       colors=['red', 'orange', 'yellow', 'green'][:len(severity_counts)])
            axes[0].set_title('Alerts by Severity', fontsize=12, fontweight='bold')
        
        if 'Status' in df_alerts.columns and len(df_alerts['Status'].unique()) > 0:
            status_counts = df_alerts['Status'].value_counts()
            axes[1].bar(range(len(status_counts)), status_counts.values, 
                       color='steelblue', alpha=0.7)
            axes[1].set_xticks(range(len(status_counts)))
            axes[1].set_xticklabels(status_counts.index, rotation=45, ha='right')
            axes[1].set_ylabel('Count', fontweight='bold')
            axes[1].set_title('Alerts by Status', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"\nNo quality alerts found for {domain_name}")

## 6. Quality Schedules Overview

In [ ]:
# Get quality schedules
if domains:
    schedules_data = run_quality_cmd(["schedules", "--account", ACCOUNT_ID, "--domainId", domain_id])
    schedules = schedules_data.get("value", [])
    
    if schedules:
        df_schedules = pd.DataFrame([{
            "Schedule Name": s.get("name", "N/A"),
            "Frequency": s.get("frequency", "N/A"),
            "Status": s.get("status", "N/A"),
            "Next Run": s.get("nextRunTime", "N/A"),
            "Last Run": s.get("lastRunTime", "N/A")
        } for s in schedules])
        
        print(f"\nQuality Schedules for {domain_name}:\n")
        display(df_schedules)
    else:
        print(f"\nNo quality schedules configured for {domain_name}")

## 7. Executive Summary Report

In [ ]:
# Generate executive summary
print("="*80)
print("DATA QUALITY EXECUTIVE SUMMARY")
print("="*80)
print(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Purview Account: {ACCOUNT_ID}")
print(f"\n{'='*80}\n")

# Overall statistics
if not df_scores.empty:
    print("OVERALL QUALITY METRICS")
    print("-" * 40)
    print(f"Total Domains Analyzed: {len(df_scores)}")
    print(f"Average Quality Score: {df_scores['Quality Score'].mean():.2f}/100")
    print(f"Average Freshness Score: {df_scores['Freshness Score'].mean():.2f}/100")
    print(f"Average Completeness Score: {df_scores['Completeness Score'].mean():.2f}/100")
    print(f"Average Accuracy Score: {df_scores['Accuracy Score'].mean():.2f}/100")
    print(f"Total Assets Across Domains: {df_scores['Total Assets'].sum()}")
    print(f"Total Assessed Assets: {df_scores['Assessed Assets'].sum()}")
    
    # Best and worst performing domains
    print(f"\n{'='*80}\n")
    print("TOP PERFORMING DOMAINS")
    print("-" * 40)
    top_domains = df_scores.nlargest(3, 'Quality Score')[['Domain', 'Quality Score']]
    for idx, row in top_domains.iterrows():
        print(f"  {row['Domain']}: {row['Quality Score']:.1f}/100")
    
    print(f"\n{'='*80}\n")
    print("DOMAINS NEEDING ATTENTION")
    print("-" * 40)
    bottom_domains = df_scores.nsmallest(3, 'Quality Score')[['Domain', 'Quality Score']]
    for idx, row in bottom_domains.iterrows():
        print(f"  {row['Domain']}: {row['Quality Score']:.1f}/100")

# Alert summary
if 'df_alerts' in locals() and not df_alerts.empty:
    print(f"\n{'='*80}\n")
    print("ACTIVE ALERTS")
    print("-" * 40)
    print(f"Total Active Alerts: {len(df_alerts)}")
    if 'Severity' in df_alerts.columns:
        for severity, count in df_alerts['Severity'].value_counts().items():
            print(f"  {severity}: {count}")

print(f"\n{'='*80}")
print("END OF REPORT")
print("="*80)

## 8. Export Results

Export analysis results to CSV files for further processing or reporting.

In [ ]:
# Export DataFrames to CSV
output_prefix = f"quality_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

if not df_scores.empty:
    df_scores.to_csv(f"{output_prefix}_domain_scores.csv", index=False)
    print(f"✓ Exported domain scores to {output_prefix}_domain_scores.csv")

if 'df_assets' in locals() and not df_assets.empty:
    df_assets.to_csv(f"{output_prefix}_assets.csv", index=False)
    print(f"✓ Exported assets to {output_prefix}_assets.csv")

if 'df_alerts' in locals() and not df_alerts.empty:
    df_alerts.to_csv(f"{output_prefix}_alerts.csv", index=False)
    print(f"✓ Exported alerts to {output_prefix}_alerts.csv")

print("\n✓ All exports completed")